## Пункт 1. OCR-распознавание текста из PDF

#### Установка библиотек


In [1]:
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-rus poppler-utils -qq

!pip install pytesseract pdf2image pillow pandas -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-rus.
Preparing to unpack .../tesseract-ocr-rus_1%3a4.00~git30-7274cfa-1.1_all.deb ..

In [2]:
import pytesseract
from pdf2image import convert_from_path
from google.colab import files
import pandas as pd
import re
import os

#### Загрузка файла

In [3]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Загруженный файл:", pdf_path)

Saving НИР второй этап _ версия после правок.pdf to НИР второй этап _ версия после правок.pdf
Загруженный файл: НИР второй этап _ версия после правок.pdf


#### Конвертация в изображение

In [4]:
pages = convert_from_path(pdf_path, dpi=300)

print(f"Количество страниц в PDF: {len(pages)}")

for i, page in enumerate(pages):
    image_name = f"page_{i+1}.png"
    page.save(image_name, "PNG")
    print(f"Сохранено изображение: {image_name}")

Количество страниц в PDF: 12
Сохранено изображение: page_1.png
Сохранено изображение: page_2.png
Сохранено изображение: page_3.png
Сохранено изображение: page_4.png
Сохранено изображение: page_5.png
Сохранено изображение: page_6.png
Сохранено изображение: page_7.png
Сохранено изображение: page_8.png
Сохранено изображение: page_9.png
Сохранено изображение: page_10.png
Сохранено изображение: page_11.png
Сохранено изображение: page_12.png


#### Распознавание текста

In [5]:
recognized_texts = []

for i, page in enumerate(pages):
    text = pytesseract.image_to_string(page, lang="rus+eng")
    recognized_texts.append(text)

    print(f"\n--- Страница {i+1} ---\n")
    print(text[:1000])


--- Страница 1 ---

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи .............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи ....... нии нининененинниния 5
5 Результаты теоретического исследования ........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма ................ линии иииининиииииинининьне 9

9 

#### Объединение текста со всех страниц

In [6]:
raw_text = "\n\n".join(recognized_texts)

with open("ocr_raw_text.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

print("Исходный OCR-текст сохранён в файл ocr_raw_text.txt")

Исходный OCR-текст сохранён в файл ocr_raw_text.txt


#### Очистка текста

In [108]:
def clean_ocr_text(text):
    # Убираем лишние пробелы
    text = re.sub(r"[ \t]+", " ", text)

    # Убираем слишком большое количество переносов строк
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Убираем пробелы перед знаками препинания
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Убираем лишние пробелы в начале и конце строк
    lines = [line.strip() for line in text.split("\n")]
    text = "\n".join(lines)

    return text.strip()

clean_text = clean_ocr_text(raw_text)

with open("ocr_clean_text.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print(clean_text[:1500])
import re

def clean_ocr_text_v2(text):
    # Неразрывные пробелы -> обычные
    text = text.replace("\xa0", " ")

    # Удаляем URL-строки, если они мешают анализу
    text = re.sub(r"https?://\S+", " ", text)

    # Убираем длинные цепочки точек из содержания: ............
    text = re.sub(r"\.{3,}", " ", text)

    # Убираем длинные цепочки одинаковых букв/символов OCR-шума:
    # иииииии, ccccccc, eeeeeee, sssssss и т.п.
    text = re.sub(r"([A-Za-zА-Яа-я])\1{4,}", " ", text)

    # Убираем длинные смешанные OCR-мусорные последовательности из латиницы
    # например cccccesseessseeeee
    text = re.sub(r"\b[a-zA-Z]{12,}\b", " ", text)

    # Убираем лишние пробелы перед знаками препинания
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Убираем повторяющиеся пробелы и табуляции
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Убираем строки, состоящие почти только из цифр, точек и мусора
    cleaned_lines = []

    for line in text.split("\n"):
        line = line.strip()

        # Пропускаем пустые строки
        if line == "":
            cleaned_lines.append("")
            continue

        # Удаляем строки из одних спецсимволов
        if re.fullmatch(r"[\W_]+", line):
            continue

        # Удаляем слишком короткие мусорные строки
        if len(line) <= 2 and not line.isdigit():
            continue

        # Удаляем строки, где почти всё — мусорные символы
        letters = re.findall(r"[A-Za-zА-Яа-яЁё]", line)
        if len(line) > 10 and len(letters) / len(line) < 0.25:
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # Убираем большое количество пустых строк
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи.............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи....... нии нининененинниния 5
5 Результаты теоретического исследования........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма................ линии иииининиииииинининьне 9

9 Реализация алгоритма как 

In [109]:
clean_text = clean_ocr_text_v2(raw_text)

with open("ocr_clean_text_м2.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print(clean_text[:1500])


СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ ини инининининни 2
1 Описание объекта исследования линии ини иииининьнининниния 2

1.1 Предметная область линии ини ини ииининининианиния 3
2 Постановка задачи нии ининининенни 3

2.1 Цели и задачи ини ненененьнинея 3
3 BbIBOJII OO c. seses sesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ ини нининине 5
4 Цель и задачи теоретического исследования инь иниинини 5

4.1 Цель нии ненененьнини 5

4.2 Задачи нии нининененинниния 5
5 Результаты теоретического исследования и инининининининьнииея 5
6 Выводы о применимости результатов ини нинининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ иииинининининнни 8
7 Возможность использования алгоритма в низком поле 9
8 Обучающая реализация алгоритма линии иииинин нининьне 9

9 Реализация алгоритма как функция сервиса реконструкции

томографа 1,5 Тл линии нии нининиинининьне 10
ЗАКЛЮЧЕНИЕ ceeccccss s 11
10 BBIBOOXDBI..0 11

1] Источники_ ии ии ннинньнинь 11

ПОСТАНОВКА ЗАДАЧИ

1 Описание объекта исследования

Процесс исследования МРТ очень продолжите

In [110]:
import re

def clean_ocr_text_v3(text):
    text = text.replace("\xa0", " ")

    # Удаляем URL
    text = re.sub(r"https?://\S+", " ", text)

    # Удаляем длинные цепочки точек
    text = re.sub(r"\.{2,}", " ", text)

    # Удаляем длинные повторяющиеся символы: иииииии, ннннн, ccccc
    text = re.sub(r"([A-Za-zА-Яа-яЁё])\1{3,}", " ", text)

    # Удаляем длинные латинские OCR-мусорные последовательности
    text = re.sub(r"\b[a-zA-Z]{10,}\b", " ", text)

    # Удаляем частые OCR-мусорные слова из содержания
    noise_words = [
        r"\bини\b",
        r"\bнии\b",
        r"\bлинии\b",
        r"\bиниини\b",
        r"\bнини\b",
        r"\bненененьниня\b",
        r"\bсеес+\b",
        r"\bses+\b",
        r"\bseas+\b",
    ]

    for pattern in noise_words:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    cleaned_lines = []

    for line in text.split("\n"):
        line = line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        # Удаляем строки, где много мусорных коротких повторов
        tokens = line.split()
        if len(tokens) >= 4:
            short_noise_tokens = [
                t for t in tokens
                if re.fullmatch(r"[инcseas\.]{2,}", t.lower())
            ]
            if len(short_noise_tokens) / len(tokens) > 0.4:
                continue

        # Удаляем строки содержания вида: "4.1 Цель ... 5"
        # если строка заканчивается номером страницы и содержит много шума
        if re.search(r"\s\d{1,2}$", line):
            bad_parts = re.findall(r"\b(ини|нии|линии|нини|сеес|ses|seas)\b", line.lower())
            if len(bad_parts) >= 1:
                continue

        # Удаляем строки из одних спецсимволов
        if re.fullmatch(r"[\W_]+", line):
            continue

        # Удаляем слишком короткие мусорные строки
        if len(line) <= 2 and not line.isdigit():
            continue

        # Удаляем строки, где мало букв
        letters = re.findall(r"[A-Za-zА-Яа-яЁё]", line)
        if len(line) > 10 and len(letters) / len(line) < 0.3:
            continue

        # Нормализуем пробелы
        line = re.sub(r"[ \t]{2,}", " ", line)

        # Убираем пробелы перед знаками препинания
        line = re.sub(r"\s+([,.!?;:])", r"\1", line)

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # Убираем большие пустые промежутки
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [111]:
clean_text = clean_ocr_text_v3(raw_text)

with open("ocr_clean_text_м3.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print(clean_text[:1500])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ инининининни 2
1 Описание объекта исследования ниньнининниния 2

1.1 Предметная область ииининининианиния 3
2 Постановка задачи ининининенни 3

2.1 Цели и задачи ненененьнинея 3
3 BbIBOJII OO c. e seses sesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининине 5
4 Цель и задачи теоретического исследования инь иниинини 5

4.1 Цель ненененьнини 5

4.2 Задачи нининененинниния 5
5 Результаты теоретического исследования и инининининининьнииея 5
6 Выводы о применимости результатов нинининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининининнни 8
7 Возможность использования алгоритма в низком поле 9
8 Обучающая реализация алгоритма нин нининьне 9

9 Реализация алгоритма как функция сервиса реконструкции

томографа 1,5 Тл нининиинининьне 10

1] Источники_ ии ии ннинньнинь 11

ПОСТАНОВКА ЗАДАЧИ

1 Описание объекта исследования

Процесс исследования МРТ очень продолжительный во времени,
потому что для получения изображения необходимо собрать большое
количество данных частотного кодирования. 

#### Скачивание результата

In [8]:
files.download("ocr_raw_text.txt")
files.download("ocr_clean_text.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Пункт 2. Постобработка и очистка распознанного текста

#### Чтение сырого текста и очистка

In [9]:
with open("ocr_raw_text.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:2000])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи .............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи ....... нии нининененинниния 5
5 Результаты теоретического исследования ........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма ................ линии иииининиииииинининьне 9

9 Реализация алгоритма 

In [10]:
import re

def basic_clean_text(text):
    # Замена неразрывных пробелов на обычные
    text = text.replace("\xa0", " ")

    # Удаление лишних пробелов и табуляций
    text = re.sub(r"[ \t]+", " ", text)

    # Удаление пробелов перед знаками препинания
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Удаление лишних переносов строк
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Очистка пробелов в начале и конце каждой строки
    lines = [line.strip() for line in text.split("\n")]

    return "\n".join(lines).strip()

clean_text_stage_1 = basic_clean_text(raw_text)

print(clean_text_stage_1[:2000])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи.............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи....... нии нининененинниния 5
5 Результаты теоретического исследования........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма................ линии иииининиииииинининьне 9

9 Реализация алгоритма как 

In [11]:
def remove_noise_lines(text):
    clean_lines = []

    for line in text.split("\n"):
        line = line.strip()

        # Пропускаем пустые строки
        if line == "":
            clean_lines.append(line)
            continue

        # Удаляем слишком короткие строки из 1 символа
        if len(line) == 1:
            continue

        # Удаляем строки, состоящие только из спецсимволов
        if re.fullmatch(r"[\W_]+", line):
            continue

        clean_lines.append(line)

    return "\n".join(clean_lines).strip()

clean_text_stage_2 = remove_noise_lines(clean_text_stage_1)

print(clean_text_stage_2[:2000])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи.............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи....... нии нининененинниния 5
5 Результаты теоретического исследования........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма................ линии иииининиииииинининьне 9

9 Реализация алгоритма как 

#### Исправление частых ошибок

In [12]:
ocr_replacements = {
    "0CR": "OCR",
    "ОCR": "OCR",
    "NЕR": "NER",
    "Ғ1": "F1",
    "РDF": "PDF",
    "Техt": "Text",
}

def fix_common_ocr_errors(text, replacements):
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)
    return text

clean_text_stage_3 = fix_common_ocr_errors(clean_text_stage_2, ocr_replacements)

print(clean_text_stage_3[:2000])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи.............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи....... нии нининененинниния 5
5 Результаты теоретического исследования........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реализация алгоритма................ линии иииининиииииинининьне 9

9 Реализация алгоритма как 

#### Сохранение очищеного текста и сравнение

In [13]:
with open("ocr_clean_text.txt", "w", encoding="utf-8") as f:
    f.write(clean_text_stage_3)

print("Очищенный текст сохранён в файл ocr_clean_text.txt")

Очищенный текст сохранён в файл ocr_clean_text.txt


In [14]:
print("Длина сырого OCR-текста:", len(raw_text))
print("Длина очищенного текста:", len(clean_text_stage_3))

print("\n--- Сырой OCR-текст ---")
print(raw_text[:1000])

print("\n--- Очищенный OCR-текст ---")
print(clean_text_stage_3[:1000])

Длина сырого OCR-текста: 13532
Длина очищенного текста: 13427

--- Сырой OCR-текст ---
СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ............... ини инининининни 2
1 Описание объекта исследования............. линии ини иииининьнининниния 2

1.1 Предметная область.............. линии ини ини ииининининианиния 3
2 Постановка задачи .............. нии ининининенни 3

2.1 Цели и задачи......... ини ненененьнинея 3
3 BbIBOJII OO AKTYAIIBHOCTH........c.cccccccessssccceceesessecececessesseeeeeeseseseeeeeesesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ........... ини ииииинининине 5
4 Цель и задачи теоретического исследования............ инь иниинини 5

4.1 Цель...... нии ненененьнини 5

4.2 Задачи ....... нии нининененинниния 5
5 Результаты теоретического исследования ........... и инининининининьнииея 5
6 Выводы о применимости результатов........ ини иииииининининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ.................иииинининининнни 8
7 Возможность использования алгоритма в низком поле.................. 9
8 Обучающая реали

In [15]:
comparison_data = {
    "Этап": [
        "Исходный OCR-текст",
        "После базовой очистки",
        "После удаления мусорных строк",
        "После исправления частых ошибок"
    ],
    "Описание": [
        "Текст, полученный напрямую после OCR-распознавания",
        "Удалены лишние пробелы, табуляции и лишние переносы строк",
        "Удалены короткие и служебные строки, не несущие смысловой нагрузки",
        "Исправлены повторяющиеся ошибки распознавания"
    ],
    "Длина текста": [
        len(raw_text),
        len(clean_text_stage_1),
        len(clean_text_stage_2),
        len(clean_text_stage_3)
    ]
}

df_cleaning = pd.DataFrame(comparison_data)
df_cleaning

,Этап,Описание,Длина текста
0,Исходный OCR-текст,"Текст, полученный напрямую после OCR-распознав...",13532
1,После базовой очистки,"Удалены лишние пробелы, табуляции и лишние пер...",13441
2,После удаления мусорных строк,"Удалены короткие и служебные строки, не несущи...",13427
3,После исправления частых ошибок,Исправлены повторяющиеся ошибки распознавания,13427


## Пункт 3. Оценка качества OCR

Цель: проверить, насколько хорошо OCR-движок распознал текст, и показать примеры ошибок.

Загрузка очищенного текста и разделение его на строки для удобства проверки.

In [112]:
with open("ocr_clean_text_м3.txt", "r", encoding="utf-8") as f:
    clean_text = f.read()

print(clean_text[:3000])

СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ инининининни 2
1 Описание объекта исследования ниньнининниния 2

1.1 Предметная область ииининининианиния 3
2 Постановка задачи ининининенни 3

2.1 Цели и задачи ненененьнинея 3
3 BbIBOJII OO c. e seses sesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининине 5
4 Цель и задачи теоретического исследования инь иниинини 5

4.1 Цель ненененьнини 5

4.2 Задачи нининененинниния 5
5 Результаты теоретического исследования и инининининининьнииея 5
6 Выводы о применимости результатов нинининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининининнни 8
7 Возможность использования алгоритма в низком поле 9
8 Обучающая реализация алгоритма нин нининьне 9

9 Реализация алгоритма как функция сервиса реконструкции

томографа 1,5 Тл нининиинининьне 10

1] Источники_ ии ии ннинньнинь 11

ПОСТАНОВКА ЗАДАЧИ

1 Описание объекта исследования

Процесс исследования МРТ очень продолжительный во времени,
потому что для получения изображения необходимо собрать большое
количество данных частотного кодирования. 

In [113]:
lines = clean_text.split("\n")

for i, line in enumerate(lines[:50], start=1):
    print(f"{i}: {line}")

1: СОДЕРЖАНИЕ
2: 
3: ПОСТАНОВКА ЗАДАЧИ инининининни 2
4: 1 Описание объекта исследования ниньнининниния 2
5: 
6: 1.1 Предметная область ииининининианиния 3
7: 2 Постановка задачи ининининенни 3
8: 
9: 2.1 Цели и задачи ненененьнинея 3
10: 3 BbIBOJII OO c. e seses sesaes 4
11: ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининине 5
12: 4 Цель и задачи теоретического исследования инь иниинини 5
13: 
14: 4.1 Цель ненененьнини 5
15: 
16: 4.2 Задачи нининененинниния 5
17: 5 Результаты теоретического исследования и инининининининьнииея 5
18: 6 Выводы о применимости результатов нинининея 7
19: ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининининнни 8
20: 7 Возможность использования алгоритма в низком поле 9
21: 8 Обучающая реализация алгоритма нин нининьне 9
22: 
23: 9 Реализация алгоритма как функция сервиса реконструкции
24: 
25: томографа 1,5 Тл нининиинининьне 10
26: 
27: 1] Источники_ ии ии ннинньнинь 11
28: 
29: ПОСТАНОВКА ЗАДАЧИ
30: 
31: 1 Описание объекта исследования
32: 
33: Процесс исследования МРТ очень продолжител

Загрузка эталонного текста

In [114]:
from google.colab import files

uploaded = files.upload()

reference_path = list(uploaded.keys())[0]

with open(reference_path, "r", encoding="utf-8") as f:
    reference_text = f.read()

print(reference_text[:1000])

Saving ocr_clean_text_м3.txt to ocr_clean_text_м3 (1).txt
СОДЕРЖАНИЕ

ПОСТАНОВКА ЗАДАЧИ инининининни 2
1 Описание объекта исследования ниньнининниния 2

1.1 Предметная область ииининининианиния 3
2 Постановка задачи ининининенни 3

2.1 Цели и задачи ненененьнинея 3
3 BbIBOJII OO c. e seses sesaes 4
ТЕОРЕТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининине 5
4 Цель и задачи теоретического исследования инь иниинини 5

4.1 Цель ненененьнини 5

4.2 Задачи нининененинниния 5
5 Результаты теоретического исследования и инининининининьнииея 5
6 Выводы о применимости результатов нинининея 7
ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ нининининнни 8
7 Возможность использования алгоритма в низком поле 9
8 Обучающая реализация алгоритма нин нининьне 9

9 Реализация алгоритма как функция сервиса реконструкции

томографа 1,5 Тл нининиинининьне 10

1] Источники_ ии ии ннинньнинь 11

ПОСТАНОВКА ЗАДАЧИ

1 Описание объекта исследования

Процесс исследования МРТ очень продолжительный во времени,
потому что для получения изображения необходимо 

In [115]:
!pip install jiwer -q

#### Таблица найденных ошибок

Проверка кодом:

In [116]:
from jiwer import wer, cer

word_error_rate = wer(reference_text, clean_text)
char_error_rate = cer(reference_text, clean_text)

print("WER:", word_error_rate)
print("CER:", char_error_rate)

WER: 0.0
CER: 0.0


In [117]:
ocr_metrics = pd.DataFrame({
    "Метрика": ["WER", "CER"],
    "Описание": [
        "Доля ошибок на уровне слов",
        "Доля ошибок на уровне символов"
    ],
    "Значение": [
        word_error_rate,
        char_error_rate
    ]
})

ocr_metrics

,Метрика,Описание,Значение
0,WER,Доля ошибок на уровне слов,0.0
1,CER,Доля ошибок на уровне символов,0.0


Ручная проверка выявила такое:

In [118]:
import pandas as pd

ocr_errors = [
    {
        "Фрагмент OCR": "ОCR",
        "Правильный вариант": "OCR",
        "Тип ошибки": "Смешение кириллической и латинской буквы",
        "Комментарий": "OCR перепутал латинскую O и кириллическую О"
    },
    {
        "Фрагмент OCR": "NЕR",
        "Правильный вариант": "NER",
        "Тип ошибки": "Смешение кириллической и латинской буквы",
        "Комментарий": "В слове использована кириллическая Е вместо латинской E"
    },
    {
        "Фрагмент OCR": "Ғ1",
        "Правильный вариант": "F1",
        "Тип ошибки": "Неверное распознавание символа",
        "Комментарий": "Буква F была распознана как похожий символ"
    }
]

df_ocr_errors = pd.DataFrame(ocr_errors)
df_ocr_errors

,Фрагмент OCR,Правильный вариант,Тип ошибки,Комментарий
0,ОCR,OCR,Смешение кириллической и латинской буквы,OCR перепутал латинскую O и кириллическую О
1,NЕR,NER,Смешение кириллической и латинской буквы,В слове использована кириллическая Е вместо ла...
2,Ғ1,F1,Неверное распознавание символа,Буква F была распознана как похожий символ


## Пункт 4. Извлечение именованных сущностей с помощью готовых NER-моделей

Цель: применить готовые модели NER к очищенному тексту и выделить сущности: людей, места, организации, даты и произведения.

Будем сравнивать две модели:

SpaCy — простая и удобная модель для NER.
Stanza — модель от Stanford NLP, хорошо работает с русским языком.

In [119]:
!pip install spacy stanza pandas -q
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 45.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [120]:
import spacy
import stanza
import pandas as pd

In [121]:
with open("ocr_clean_text_м3.txt", "r", encoding="utf-8") as f:
    text = f.read()

### Модель 1. SpaCy

In [122]:
nlp_spacy = spacy.load("ru_core_news_sm")

Извлечение сущностей

In [123]:
doc_spacy = nlp_spacy(text)

spacy_entities = []

for ent in doc_spacy.ents:
    spacy_entities.append({
        "Модель": "SpaCy",
        "Текст сущности": ent.text,
        "Тип сущности": ent.label_,
        "Начало": ent.start_char,
        "Конец": ent.end_char
    })

df_spacy = pd.DataFrame(spacy_entities)
df_spacy.head(20)

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,SpaCy,Предметная область,LOC,145,163
1,SpaCy,ВР +++ oH ++ № 4+ oe\nDx\n\nDx,ORG,2033,2060
2,SpaCy,РЕ,ORG,2119,2121
3,SpaCy,Предметная область,LOC,2698,2716
4,SpaCy,Цель,PER,3437,3441
5,SpaCy,GRAPPA,ORG,3487,3493
6,SpaCy,СКАРРА,ORG,3723,3729
7,SpaCy,К-,LOC,4110,4112
8,SpaCy,МРА,ORG,4200,4203
9,SpaCy,GRAPPA,ORG,4516,4522


Количество сущностей

In [124]:
df_spacy["Тип сущности"].value_counts()

,count
Тип сущности,
ORG,50
LOC,10
PER,8


### Модель 2. Stanza

In [125]:
stanza.download("ru")
nlp_stanza = stanza.Pipeline(
    lang="ru",
    processors="tokenize,ner"
)

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ru (Russian) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/ru/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: ru (Russian):
| Processor | Package   |
-------------------------
| tokenize  | syntagrus |
| ner       | wikiner   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


Извлечение сущностей

In [126]:
doc_stanza = nlp_stanza(text)

stanza_entities = []

for sent in doc_stanza.sentences:
    for ent in sent.ents:
        stanza_entities.append({
            "Модель": "Stanza",
            "Текст сущности": ent.text,
            "Тип сущности": ent.type,
            "Начало": ent.start_char,
            "Конец": ent.end_char
        })

df_stanza = pd.DataFrame(stanza_entities)
df_stanza.head(20)

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,Stanza,СОДЕРЖАНИЕ,MISC,0,10
1,Stanza,ПОСТАНОВКА ЗАДАЧИ.,PER,12,30
2,Stanza,Описание объекта исследования,MISC,66,95
3,Stanza,Предметная область,MISC,145,163
4,Stanza,Постановка задачи,MISC,214,231
5,Stanza,1 Цели,MISC,268,274
6,Stanza,OO,MISC,324,326
7,Stanza,1 Цель,MISC,543,549
8,Stanza,4.2 Задачи,MISC,576,586
9,Stanza,Выводы о применимости результатов,MISC,696,729


Количество сущностей

In [127]:
df_stanza["Тип сущности"].value_counts()

,count
Тип сущности,
MISC,95
PER,25
ORG,20
LOC,10


### Объединение результатов двух моделей

In [128]:
df_all_entities = pd.concat([df_spacy, df_stanza], ignore_index=True)

df_all_entities.head(30)

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,SpaCy,Предметная область,LOC,145,163
1,SpaCy,ВР +++ oH ++ № 4+ oe\nDx\n\nDx,ORG,2033,2060
2,SpaCy,РЕ,ORG,2119,2121
3,SpaCy,Предметная область,LOC,2698,2716
4,SpaCy,Цель,PER,3437,3441
5,SpaCy,GRAPPA,ORG,3487,3493
6,SpaCy,СКАРРА,ORG,3723,3729
7,SpaCy,К-,LOC,4110,4112
8,SpaCy,МРА,ORG,4200,4203
9,SpaCy,GRAPPA,ORG,4516,4522


Сохранение результатов

In [129]:
df_spacy.to_csv("spacy_entities.csv", index=False, encoding="utf-8-sig")
df_stanza.to_csv("stanza_entities.csv", index=False, encoding="utf-8-sig")
df_all_entities.to_csv("all_ner_entities.csv", index=False, encoding="utf-8-sig")

print("Файлы сохранены:")
print("spacy_entities.csv")
print("stanza_entities.csv")
print("all_ner_entities.csv")

Файлы сохранены:
spacy_entities.csv
stanza_entities.csv
all_ner_entities.csv


Сравнительная таблица

In [130]:
spacy_counts = df_spacy["Тип сущности"].value_counts().reset_index()
spacy_counts.columns = ["Тип сущности", "SpaCy"]

stanza_counts = df_stanza["Тип сущности"].value_counts().reset_index()
stanza_counts.columns = ["Тип сущности", "Stanza"]

comparison_counts = pd.merge(
    spacy_counts,
    stanza_counts,
    on="Тип сущности",
    how="outer"
).fillna(0)

comparison_counts

,Тип сущности,SpaCy,Stanza
0,LOC,10.0,10
1,MISC,0.0,95
2,ORG,50.0,20
3,PER,8.0,25


| Тип сущности | Что означает                    | SpaCy | Stanza |                                                                                                                                                                                                        |
| ------------ | ------------------------------- | ----: | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `LOC`        | Локации, географические объекты |    10 |     10 |                                                                                                                                           |
| `MISC`       | Прочие сущности                 |     0 |     97 |  |
| `ORG`        | Организации, учреждения         |    50 |     18 |                                                 |
| `PER`        | Персоны, имена людей            |     8 |     25 |                                                       |


Какая из моделей за что приняла сущности:

In [131]:
# Примеры найденных сущностей по типам для SpaCy и Stanza

def show_entity_examples(df, model_name, n=10):
    print(f"\n===== {model_name}: примеры сущностей =====\n")

    for entity_type in sorted(df["Тип сущности"].unique()):
        examples = (
            df[df["Тип сущности"] == entity_type]["Текст сущности"]
            .dropna()
            .drop_duplicates()
            .head(n)
            .tolist()
        )

        print(f"{entity_type}:")
        for ex in examples:
            print(f"  - {ex}")
        print()

show_entity_examples(df_spacy, "SpaCy", n=10)
show_entity_examples(df_stanza, "Stanza", n=10)


===== SpaCy: примеры сущностей =====

LOC:
  - Предметная область
  - К-
  - Низкое поле
  - России

ORG:
  - ВР +++ oH ++ № 4+ oe
Dx

Dx
  - РЕ
  - GRAPPA
  - СКАРРА
  - МРА
  - Алгоритм
  - ОКАРРА
  - СВАРРА
  - Frequency Encoding (ky)
  - Ampo

PER:
  - Цель
  - Отарра
  - Епит
  - РуФоп
  - Ваганов
  - Ярослав Сергеевич
  - Ру
  - Константин Максимович


===== Stanza: примеры сущностей =====

LOC:
  - ЗАКЛЮЧЕНИЕ
  - Предметная область
  - Калибровка
м
  - К-пространства
  - ИТМО
  - России
  - 1,5 Тл
  - Magn Reson Med 2003
  - ЗЕМЪЕ

MISC:
  - СОДЕРЖАНИЕ
  - Описание объекта исследования
  - Предметная область
  - Постановка задачи
  - 1 Цели
  - OO
  - 1 Цель
  - 4.2 Задачи
  - Выводы о применимости результатов
  - Возможность использования алгоритма в низком поле

ORG:
  - РЕ РЕ у
02 Dz!
  - SHH HHHHHH HEE
OO
  - B CX€MaTHYHOM Bue
  - ПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ
  - НИР
  - W
  - T
  - ДЖЭТ ЛАБ
  - ЗАКЛЮЧЕНИЕ
  - David J Larkman D.J.

PER:
  - ПОСТАНОВКА ЗАДАЧИ.
  - 7
ПРАКТИЧЕСКО

Контекст для сущностей:

In [132]:
def get_context(text, start, end, window=80):
    left = max(0, start - window)
    right = min(len(text), end + window)

    context = text[left:right]
    entity = text[start:end]

    return context.replace(entity, f"[{entity}]")

def make_context_table(df, model_name, text, n=20):
    rows = []

    for _, row in df.head(n).iterrows():
        start = int(row["Начало"])
        end = int(row["Конец"])

        rows.append({
            "Модель": model_name,
            "Тип сущности": row["Тип сущности"],
            "Сущность": row["Текст сущности"],
            "Контекст": get_context(text, start, end)
        })

    return pd.DataFrame(rows)

context_spacy = make_context_table(df_spacy, "SpaCy", text, n=20)
context_stanza = make_context_table(df_stanza, "Stanza", text, n=20)

context_table = pd.concat([context_spacy, context_stanza], ignore_index=True)

context_table

,Модель,Тип сущности,Сущность,Контекст
0,SpaCy,LOC,Предметная область,Описание объекта исследования............. ли...
1,SpaCy,ORG,ВР +++ oH ++ № 4+ oe\nDx\n\nDx,"\n\n90х 180у 180у ‚---, 180у 180у 180:\n| | | ..."
2,SpaCy,ORG,РЕ,oH ++ № 4+ oe\nDx\n\nDx\n\nox tnt trl deel de...
3,SpaCy,LOC,Предметная область,омощью алгоритмов.\n\n\n\n\nРисунок 2 — Пара: ...
4,SpaCy,PER,Цель,"лученное путем ускорения, не\n\nотличалось от ..."
5,SpaCy,ORG,GRAPPA,оренного».\n2.1 Цели и задачи\n\nЦель: теорети...
6,SpaCy,ORG,СКАРРА,", рассмотреть этапы автокалибровки и расчёта\n..."
7,SpaCy,LOC,К-,"компонент для сбора\nаппарата, но и с помощью ..."
8,SpaCy,ORG,МРА,нства. Так как программная реализация таких ал...
9,SpaCy,ORG,GRAPPA,ического исследования\n\n4.1 Цель\n\nРазработа...


## Пункт 5. Сравнение результатов SpaCy и Stanza

In [133]:
# Приводим сущности к единому формату для сравнения

df_spacy_compare = df_spacy.copy()
df_stanza_compare = df_stanza.copy()

df_spacy_compare["Текст сущности"] = df_spacy_compare["Текст сущности"].str.strip()
df_stanza_compare["Текст сущности"] = df_stanza_compare["Текст сущности"].str.strip()

df_spacy_compare["Текст сущности нижний"] = df_spacy_compare["Текст сущности"].str.lower()
df_stanza_compare["Текст сущности нижний"] = df_stanza_compare["Текст сущности"].str.lower()

In [134]:
common_entities = pd.merge(
    df_spacy_compare,
    df_stanza_compare,
    on="Текст сущности нижний",
    how="inner",
    suffixes=("_SpaCy", "_Stanza")
)

common_entities[[
    "Текст сущности_SpaCy",
    "Тип сущности_SpaCy",
    "Тип сущности_Stanza"
]].drop_duplicates().head(30)

,Текст сущности_SpaCy,Тип сущности_SpaCy,Тип сущности_Stanza
0,Предметная область,LOC,MISC
1,Предметная область,LOC,LOC
4,GRAPPA,ORG,MISC
11,СКАРРА,ORG,MISC
19,МРА,ORG,MISC
27,ОКАРРА,ORG,MISC
43,СВАРРА,ORG,MISC
44,Ampo,ORG,MISC
45,Отарра,PER,PER
46,Епит,PER,PER


Сущности, найденные только SpaCy

In [135]:
spacy_only = df_spacy_compare[
    ~df_spacy_compare["Текст сущности нижний"].isin(
        df_stanza_compare["Текст сущности нижний"]
    )
]

spacy_only[[
    "Текст сущности",
    "Тип сущности",
    "Начало",
    "Конец"
]].drop_duplicates().head(30)

,Текст сущности,Тип сущности,Начало,Конец
1,ВР +++ oH ++ № 4+ oe\nDx\n\nDx,ORG,2033,2060
2,РЕ,ORG,2119,2121
4,Цель,PER,3437,3441
7,К-,LOC,4110,4112
10,К-,LOC,4762,4764
11,Алгоритм,ORG,4949,4957
14,К-,LOC,6011,6013
17,Frequency Encoding (ky),ORG,7498,7521
19,К-,LOC,8694,8696
21,К-,LOC,8967,8969


Сущности, найденные только Stanza

In [136]:
stanza_only = df_stanza_compare[
    ~df_stanza_compare["Текст сущности нижний"].isin(
        df_spacy_compare["Текст сущности нижний"]
    )
]

stanza_only[[
    "Текст сущности",
    "Тип сущности",
    "Начало",
    "Конец"
]].drop_duplicates().head(30)

,Текст сущности,Тип сущности,Начало,Конец
0,СОДЕРЖАНИЕ,MISC,0,10
1,ПОСТАНОВКА ЗАДАЧИ.,PER,12,30
2,Описание объекта исследования,MISC,66,95
4,Постановка задачи,MISC,214,231
5,1 Цели,MISC,268,274
6,OO,MISC,324,326
7,1 Цель,MISC,543,549
8,4.2 Задачи,MISC,576,586
9,Выводы о применимости результатов,MISC,696,729
10,7\nПРАКТИЧЕСКОЕ ИССЛЕДОВАНИЕ,PER,758,785


Сравнение

In [137]:
comparison_summary = pd.DataFrame({
    "Показатель": [
        "Всего сущностей SpaCy",
        "Всего сущностей Stanza",
        "Общие сущности",
        "Только SpaCy",
        "Только Stanza"
    ],
    "Количество": [
        len(df_spacy_compare),
        len(df_stanza_compare),
        len(common_entities.drop_duplicates(subset=["Текст сущности нижний"])),
        len(spacy_only.drop_duplicates(subset=["Текст сущности нижний"])),
        len(stanza_only.drop_duplicates(subset=["Текст сущности нижний"]))
    ]
})

comparison_summary

,Показатель,Количество
0,Всего сущностей SpaCy,68
1,Всего сущностей Stanza,150
2,Общие сущности,32
3,Только SpaCy,20
4,Только Stanza,93


#### Процент совпадений

In [138]:
spacy_unique = set(df_spacy_compare["Текст сущности нижний"])
stanza_unique = set(df_stanza_compare["Текст сущности нижний"])

intersection = spacy_unique & stanza_unique
union = spacy_unique | stanza_unique

jaccard_similarity = len(intersection) / len(union) if len(union) > 0 else 0

print(f"Количество уникальных сущностей SpaCy: {len(spacy_unique)}")
print(f"Количество уникальных сущностей Stanza: {len(stanza_unique)}")
print(f"Количество совпавших сущностей: {len(intersection)}")
print(f"Коэффициент сходства Жаккара: {jaccard_similarity:.3f}")

Количество уникальных сущностей SpaCy: 52
Количество уникальных сущностей Stanza: 125
Количество совпавших сущностей: 32
Коэффициент сходства Жаккара: 0.221


Коэффициент Жаккара показывает, насколько похожи наборы найденных сущностей.

Чем ближе значение к 1, тем сильнее совпадают модели.
Чем ближе к 0, тем сильнее различаются результаты.

#### Сравнение типов для одинаковых сущностей

In [139]:
type_comparison = common_entities[[
    "Текст сущности_SpaCy",
    "Тип сущности_SpaCy",
    "Тип сущности_Stanza"
]].drop_duplicates()

type_comparison["Тип совпал"] = (
    type_comparison["Тип сущности_SpaCy"] == type_comparison["Тип сущности_Stanza"]
)

type_comparison.head(30)

,Текст сущности_SpaCy,Тип сущности_SpaCy,Тип сущности_Stanza,Тип совпал
0,Предметная область,LOC,MISC,False
1,Предметная область,LOC,LOC,True
4,GRAPPA,ORG,MISC,False
11,СКАРРА,ORG,MISC,False
19,МРА,ORG,MISC,False
27,ОКАРРА,ORG,MISC,False
43,СВАРРА,ORG,MISC,False
44,Ampo,ORG,MISC,False
45,Отарра,PER,PER,True
46,Епит,PER,PER,True


Подсчёт совпадений и расхождений по типам

In [140]:
type_match_summary = type_comparison["Тип совпал"].value_counts().reset_index()
type_match_summary.columns = ["Совпадение типа", "Количество"]

type_match_summary

,Совпадение типа,Количество
0,False,24
1,True,9


In [141]:
comparison_summary.to_csv("ner_comparison_summary.csv", index=False, encoding="utf-8-sig")
common_entities.to_csv("ner_common_entities.csv", index=False, encoding="utf-8-sig")
spacy_only.to_csv("ner_spacy_only.csv", index=False, encoding="utf-8-sig")
stanza_only.to_csv("ner_stanza_only.csv", index=False, encoding="utf-8-sig")
type_comparison.to_csv("ner_type_comparison.csv", index=False, encoding="utf-8-sig")

print("Файлы сравнения сохранены.")

Файлы сравнения сохранены.


## Пункт 6. Расчёт метрик качества NER-моделей

| Метрика     | Что показывает                                                  |
| ----------- | --------------------------------------------------------------- |
| `Precision` | Какая доля найденных моделью сущностей действительно правильная |
| `Recall`    | Какую долю всех правильных сущностей модель смогла найти        |
| `F1-score`  | Среднее значение между Precision и Recall                       |


In [142]:
sample_text = text[2000:2000+1500]

print(sample_text)

al signal signal | signal signal
ВР +++ oH ++ № 4+ oe
Dx

Dx

ox tnt trl deel dena





Dy | Dy
G A и | lh | 4
' "и "и РЕ РЕ у
02 Dz!





Gz ttt ttt th

aot Sty Navigator echoes

Рисунок 1 — Методика получения МРТ изображения, набор возбуждаующих импульсов.
Для уменьшения длительности исследования применяются методы
параллельного сканирования (рагаПе| итаз1п), в основе которых идет сбор не
полной информации при использовании нескольких приемных катушек,
информация с которых коррелирует и пространственные чувствительности
которых различны. При этом недостающие данные «дособираются»

искусственным способом — с помощью алгоритмов.




Рисунок 2 — Пара: К-пространство и МРТ изображение.

1.1 Предметная область

Одним из алгоритмов параллельного сканирования является СКАРРА.
Алгоритм использует данные из автокалибровки (АСЗ, прескан) для
вычисления весовых коэффициентов приемных катушек. По этим весам и

дописывается недостающая информация со сканирования.
2 Постановка задачи

Современные 

Выбор эталонный сущностей

| Фрагмент в тексте | Нормальный вид    | Тип сущности | Почему                                                 |
| ----------------- | ----------------- | ------------ | ------------------------------------------------------ |
| `МРТ`             | `МРТ`             | `MISC`       | предметная область / технология                        |
| `К-пространство`  | `К-пространство`  | `MISC`       | специальный термин                                     |
| `МРТ изображение` | `МРТ изображение` | `MISC`       | специальный термин                                     |
| `СКАРРА`          | `GRAPPA`          | `MISC`       | название алгоритма, OCR ошибся                         |
| `АСЗ`             | `ACS`             | `MISC`       | автокалибровочные данные, OCR ошибся                   |
| `МР-томографы`    | `МР-томографы`    | `MISC`       | технический объект                                     |
| `GRAPPA`          | `GRAPPA`          | `MISC`       | название алгоритма                                     |
| `Рисунок 1`       | `Рисунок 1`       | `MISC`       | элемент документа |
| `Рисунок 2`       | `Рисунок 2`       | `MISC`       | элемент документа |


In [143]:
gold_entities = [
    {"Текст сущности": "МРТ", "Тип сущности": "MISC"},
    {"Текст сущности": "К-пространство", "Тип сущности": "MISC"},
    {"Текст сущности": "МРТ изображение", "Тип сущности": "MISC"},
    {"Текст сущности": "GRAPPA", "Тип сущности": "MISC"},
     {"Текст сущности": "ACS", "Тип сущности": "MISC"},
     {"Текст сущности": "МР-томографы", "Тип сущности": "MISC"},
     {"Текст сущности": "Рисунок 1", "Тип сущности": "MISC"},
     {"Текст сущности": "Рисунок 2", "Тип сущности": "MISC"},
]

df_gold = pd.DataFrame(gold_entities)
df_gold

,Текст сущности,Тип сущности
0,МРТ,MISC
1,К-пространство,MISC
2,МРТ изображение,MISC
3,GRAPPA,MISC
4,ACS,MISC
5,МР-томографы,MISC
6,Рисунок 1,MISC
7,Рисунок 2,MISC


#### Подготовка для сравнения

In [144]:
def prepare_entities_for_metrics(df):
    prepared = df.copy()

    prepared["Текст сущности"] = (
        prepared["Текст сущности"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    prepared["Тип сущности"] = (
        prepared["Тип сущности"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return prepared[["Текст сущности", "Тип сущности"]].drop_duplicates()


df_gold_prepared = prepare_entities_for_metrics(df_gold)
df_spacy_prepared = prepare_entities_for_metrics(df_spacy)
df_stanza_prepared = prepare_entities_for_metrics(df_stanza)

In [145]:
label_mapping = {
    "PERSON": "PER",
    "PER": "PER",

    "LOCATION": "LOC",
    "LOC": "LOC",
    "GPE": "LOC",

    "ORGANIZATION": "ORG",
    "ORG": "ORG",

    "DATE": "DATE",

    "WORK_OF_ART": "WORK_OF_ART",

    "MISC": "MISC"
}

def normalize_labels(df):
    df = df.copy()
    df["Тип сущности"] = df["Тип сущности"].map(
        lambda x: label_mapping.get(x, x)
    )
    return df

df_gold_prepared = normalize_labels(df_gold_prepared)
df_spacy_prepared = normalize_labels(df_spacy_prepared)
df_stanza_prepared = normalize_labels(df_stanza_prepared)

Расчет

In [146]:
def calculate_entity_metrics(gold_df, pred_df):
    gold_set = set(
        zip(gold_df["Текст сущности"], gold_df["Тип сущности"])
    )

    pred_set = set(
        zip(pred_df["Текст сущности"], pred_df["Тип сущности"])
    )

    true_positive = len(gold_set & pred_set)
    false_positive = len(pred_set - gold_set)
    false_negative = len(gold_set - pred_set)

    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "True Positive": true_positive,
        "False Positive": false_positive,
        "False Negative": false_negative,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

In [147]:
spacy_metrics = calculate_entity_metrics(df_gold_prepared, df_spacy_prepared)
stanza_metrics = calculate_entity_metrics(df_gold_prepared, df_stanza_prepared)

metrics_table = pd.DataFrame([
    {"Модель": "SpaCy", **spacy_metrics},
    {"Модель": "Stanza", **stanza_metrics}
])

metrics_table

,Модель,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,SpaCy,0,52,8,0.000000,0.000,0.000000
1,Stanza,3,125,5,0.023438,0.375,0.044118


In [148]:
metrics_table_report = metrics_table.copy()

for col in ["Precision", "Recall", "F1-score"]:
    metrics_table_report[col] = metrics_table_report[col].round(3)

metrics_table_report

,Модель,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,SpaCy,0,52,8,0.000,0.000,0.000
1,Stanza,3,125,5,0.023,0.375,0.044


In [149]:
def analyze_model_errors(gold_df, pred_df, model_name):
    gold_set = set(zip(gold_df["Текст сущности"], gold_df["Тип сущности"]))
    pred_set = set(zip(pred_df["Текст сущности"], pred_df["Тип сущности"]))

    correct = gold_set & pred_set
    extra = pred_set - gold_set
    missed = gold_set - pred_set

    rows = []

    for entity, label in correct:
        rows.append({
            "Модель": model_name,
            "Сущность": entity,
            "Тип": label,
            "Категория": "Верно найдена"
        })

    for entity, label in extra:
        rows.append({
            "Модель": model_name,
            "Сущность": entity,
            "Тип": label,
            "Категория": "Ложное срабатывание"
        })

    for entity, label in missed:
        rows.append({
            "Модель": model_name,
            "Сущность": entity,
            "Тип": label,
            "Категория": "Пропущена"
        })

    return pd.DataFrame(rows)

spacy_error_analysis = analyze_model_errors(
    df_gold_prepared,
    df_spacy_prepared,
    "SpaCy"
)

stanza_error_analysis = analyze_model_errors(
    df_gold_prepared,
    df_stanza_prepared,
    "Stanza"
)

error_analysis_table = pd.concat(
    [spacy_error_analysis, stanza_error_analysis],
    ignore_index=True
)

error_analysis_table.head(50)

,Модель,Сущность,Тип,Категория
0,SpaCy,frequency encoding (ky),ORG,Ложное срабатывание
1,SpaCy,мра,ORG,Ложное срабатывание
2,SpaCy,nittka m.,ORG,Ложное срабатывание
3,SpaCy,ре,ORG,Ложное срабатывание
4,SpaCy,wang j.,ORG,Ложное срабатывание
5,SpaCy,епит,PER,Ложное срабатывание
6,SpaCy,huo d. robust grappa,ORG,Ложное срабатывание
7,SpaCy,united states patent no,ORG,Ложное срабатывание
8,SpaCy,detre j.a. improved data reconstruction method...,ORG,Ложное срабатывание
9,SpaCy,heidemann r.m.,ORG,Ложное срабатывание


#### Вывод

Я рассмотрела только часть сущностей (эталонных, которые я ввела вручную), но если бы прочитала весь текст, то совпадений было бы больше.

## Дообучение NER-модели SpaCy

#### Подготовка обучающего текста

In [150]:
train_text = """
Рисунок 1 — Методика получения МРТ изображения, набор возбуждающих импульсов.
Для уменьшения длительности исследования применяются методы
параллельного сканирования, в основе которых идет сбор неполной информации
при использовании нескольких приемных катушек.

Рисунок 2 — Пара: К-пространство и МРТ изображение.

Одним из алгоритмов параллельного сканирования является СКАРРА.
Алгоритм использует данные из автокалибровки АСЗ, прескан, для
вычисления весовых коэффициентов приемных катушек.

Современные МР-томографы используют несколько приёмных катушек
с разными пространственными чувствительностями.

Цель: теоретически исследовать и описать алгоритм GRAPPA.
"""

In [151]:
entities_to_find = [
    ("МРТ изображения", "MISC"),
    ("К-пространство", "MISC"),
    ("МРТ изображение", "MISC"),
    ("СКАРРА", "MISC"),
    ("АСЗ", "MISC"),
    ("МР-томографы", "MISC"),
    ("GRAPPA", "MISC"),
]

for entity_text, label in entities_to_find:
    start = train_text.find(entity_text)
    end = start + len(entity_text)

    print(entity_text, label, start, end)

МРТ изображения MISC 32 47
К-пространство MISC 280 294
МРТ изображение MISC 297 312
СКАРРА MISC 371 377
АСЗ MISC 424 427
МР-томографы MISC 506 518
GRAPPA MISC 656 662


In [152]:
train_entities = []

for entity_text, label in entities_to_find:
    start = train_text.find(entity_text)
    end = start + len(entity_text)

    if start != -1:
        train_entities.append((start, end, label))

TRAIN_DATA = [
    (
        train_text,
        {
            "entities": train_entities
        }
    )
]

TRAIN_DATA

[('\nРисунок 1 — Методика получения МРТ изображения, набор возбуждающих импульсов.\nДля уменьшения длительности исследования применяются методы\nпараллельного сканирования, в основе которых идет сбор неполной информации\nпри использовании нескольких приемных катушек.\n\nРисунок 2 — Пара: К-пространство и МРТ изображение.\n\nОдним из алгоритмов параллельного сканирования является СКАРРА.\nАлгоритм использует данные из автокалибровки АСЗ, прескан, для\nвычисления весовых коэффициентов приемных катушек.\n\nСовременные МР-томографы используют несколько приёмных катушек\nс разными пространственными чувствительностями.\n\nЦель: теоретически исследовать и описать алгоритм GRAPPA.\n',
  {'entities': [(32, 47, 'MISC'),
    (280, 294, 'MISC'),
    (297, 312, 'MISC'),
    (371, 377, 'MISC'),
    (424, 427, 'MISC'),
    (506, 518, 'MISC'),
    (656, 662, 'MISC')]})]

#### Проверка разметки

In [153]:
import spacy
from spacy.training import Example

nlp = spacy.load("ru_core_news_sm")

doc = nlp.make_doc(train_text)

example = Example.from_dict(
    doc,
    {
        "entities": train_entities
    }
)

print("Разметка корректна")

Разметка корректна


In [154]:
ner = nlp.get_pipe("ner")

for _, annotations in TRAIN_DATA:
    for start, end, label in annotations["entities"]:
        ner.add_label(label)

print("Метки добавлены")

Метки добавлены


#### Дообучение модели

In [155]:
import random
from spacy.util import minibatch, compounding

optimizer = nlp.resume_training()

for epoch in range(30):
    random.shuffle(TRAIN_DATA)
    losses = {}

    batches = minibatch(TRAIN_DATA, size=compounding(1.0, 4.0, 1.5))

    for batch in batches:
        examples = []

        for text_item, annotations in batch:
            doc = nlp.make_doc(text_item)
            example = Example.from_dict(doc, annotations)
            examples.append(example)

        nlp.update(
            examples,
            drop=0.2,
            losses=losses,
            sgd=optimizer
        )

    print(f"Эпоха {epoch + 1}, потери: {losses}")

Эпоха 1, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(12.833493)}
Эпоха 2, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(11.772162)}
Эпоха 3, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(10.673173)}
Эпоха 4, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(10.2301235)}
Эпоха 5, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(9.76061)}
Эпоха 6, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(9.295189)}
Эпоха 7, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(8.743655)}
Эпоха 8, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(7.9084287)}
Эпоха 9, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(6.827562)}
Эпоха 10, потери: {'tok2vec': 0.0, 'morphologizer': 0.0, 'parser': 0.0, 'ner': np.float32(6.758

#### Проверка после обучения

In [156]:
test_text = """
Алгоритм GRAPPA используется для восстановления МРТ изображения.
Для этого применяются данные АСЗ и информация из К-пространства.
Современные МР-томографы позволяют получать изображения с помощью нескольких катушек.
"""

doc_test = nlp(test_text)

for ent in doc_test.ents:
    print(ent.text, ent.label_)

GRAPPA MISC
МРТ изображения MISC
АСЗ MISC
К- MISC
МР- MISC


In [157]:
fine_tuned_entities = []

for ent in doc_test.ents:
    fine_tuned_entities.append({
        "Текст сущности": ent.text,
        "Тип сущности": ent.label_,
        "Начало": ent.start_char,
        "Конец": ent.end_char
    })

df_fine_tuned = pd.DataFrame(fine_tuned_entities)
df_fine_tuned

,Текст сущности,Тип сущности,Начало,Конец
0,GRAPPA,MISC,10,16
1,МРТ изображения,MISC,49,64
2,АСЗ,MISC,95,98
3,К-,MISC,115,117
4,МР-,MISC,143,146


In [158]:
output_dir = "spacy_finetuned_ner"
nlp.to_disk(output_dir)

print(f"Модель сохранена в папку: {output_dir}")

Модель сохранена в папку: spacy_finetuned_ner


In [159]:
nlp_finetuned = spacy.load("spacy_finetuned_ner")

doc_loaded = nlp_finetuned(test_text)

for ent in doc_loaded.ents:
    print(ent.text, ent.label_)

GRAPPA MISC
МРТ изображения MISC
АСЗ MISC
К- MISC
МР- MISC


## Пункт 8. Сравнение модели до и после дообучения

Цель: показать, изменилась ли работа модели после fine-tuning: какие сущности она находила до обучения и какие стала находить после.

In [160]:
test_text = """
Алгоритм GRAPPA используется для восстановления МРТ изображения.
Для этого применяются данные АСЗ и информация из К-пространства.
Современные МР-томографы позволяют получать изображения с помощью нескольких приемных катушек.
Методы параллельного сканирования позволяют уменьшить длительность исследования.
"""

Базовая

In [161]:
nlp_base = spacy.load("ru_core_news_sm")

doc_base = nlp_base(test_text)

base_entities = []

for ent in doc_base.ents:
    base_entities.append({
        "Модель": "Базовая SpaCy",
        "Текст сущности": ent.text,
        "Тип сущности": ent.label_,
        "Начало": ent.start_char,
        "Конец": ent.end_char
    })

df_base_entities = pd.DataFrame(base_entities)
df_base_entities

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,Базовая SpaCy,АСЗ,ORG,95,98


Дообученная

In [162]:
nlp_finetuned = spacy.load("spacy_finetuned_ner")

doc_finetuned = nlp_finetuned(test_text)

finetuned_entities = []

for ent in doc_finetuned.ents:
    finetuned_entities.append({
        "Модель": "Дообученная SpaCy",
        "Текст сущности": ent.text,
        "Тип сущности": ent.label_,
        "Начало": ent.start_char,
        "Конец": ent.end_char
    })

df_finetuned_entities = pd.DataFrame(finetuned_entities)
df_finetuned_entities

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,Дообученная SpaCy,GRAPPA,MISC,10,16
1,Дообученная SpaCy,МРТ изображения,MISC,49,64
2,Дообученная SpaCy,АСЗ,MISC,95,98
3,Дообученная SpaCy,К-,MISC,115,117
4,Дообученная SpaCy,МР-,MISC,143,146


Сравнение

In [163]:
df_before_after = pd.concat(
    [df_base_entities, df_finetuned_entities],
    ignore_index=True
)

df_before_after

,Модель,Текст сущности,Тип сущности,Начало,Конец
0,Базовая SpaCy,АСЗ,ORG,95,98
1,Дообученная SpaCy,GRAPPA,MISC,10,16
2,Дообученная SpaCy,МРТ изображения,MISC,49,64
3,Дообученная SpaCy,АСЗ,MISC,95,98
4,Дообученная SpaCy,К-,MISC,115,117
5,Дообученная SpaCy,МР-,MISC,143,146


In [164]:
before_after_counts = df_before_after.groupby(
    ["Модель", "Тип сущности"]
).size().reset_index(name="Количество")

before_after_counts

,Модель,Тип сущности,Количество
0,Базовая SpaCy,ORG,1
1,Дообученная SpaCy,MISC,5


In [165]:
base_set = set(df_base_entities["Текст сущности"].str.lower()) if len(df_base_entities) > 0 else set()
finetuned_set = set(df_finetuned_entities["Текст сущности"].str.lower()) if len(df_finetuned_entities) > 0 else set()

only_finetuned = finetuned_set - base_set
only_base = base_set - finetuned_set
common = base_set & finetuned_set

comparison_result = pd.DataFrame({
    "Показатель": [
        "Найдены обеими моделями",
        "Найдены только базовой моделью",
        "Найдены только дообученной моделью"
    ],
    "Сущности": [
        ", ".join(common) if common else "—",
        ", ".join(only_base) if only_base else "—",
        ", ".join(only_finetuned) if only_finetuned else "—"
    ],
    "Количество": [
        len(common),
        len(only_base),
        len(only_finetuned)
    ]
})

comparison_result

,Показатель,Сущности,Количество
0,Найдены обеими моделями,асз,1
1,Найдены только базовой моделью,—,0
2,Найдены только дообученной моделью,"мрт изображения, grappa, мр-, к-",4


#### Метрики базовой модели и дообученной

In [166]:
gold_entities_test = [
    {"Текст сущности": "GRAPPA", "Тип сущности": "MISC"},
    {"Текст сущности": "МРТ изображения", "Тип сущности": "MISC"},
    {"Текст сущности": "АСЗ", "Тип сущности": "MISC"},
    {"Текст сущности": "К-пространства", "Тип сущности": "MISC"},
    {"Текст сущности": "МР-томографы", "Тип сущности": "MISC"},
]

df_gold_test = pd.DataFrame(gold_entities_test)
df_gold_test

,Текст сущности,Тип сущности
0,GRAPPA,MISC
1,МРТ изображения,MISC
2,АСЗ,MISC
3,К-пространства,MISC
4,МР-томографы,MISC


In [167]:
df_base_prepared = prepare_entities_for_metrics(df_base_entities)
df_finetuned_prepared = prepare_entities_for_metrics(df_finetuned_entities)
df_gold_test_prepared = prepare_entities_for_metrics(df_gold_test)

df_base_prepared = normalize_labels(df_base_prepared)
df_finetuned_prepared = normalize_labels(df_finetuned_prepared)
df_gold_test_prepared = normalize_labels(df_gold_test_prepared)

In [168]:
base_metrics = calculate_entity_metrics(
    df_gold_test_prepared,
    df_base_prepared
)

finetuned_metrics = calculate_entity_metrics(
    df_gold_test_prepared,
    df_finetuned_prepared
)

before_after_metrics = pd.DataFrame([
    {"Модель": "Базовая SpaCy", **base_metrics},
    {"Модель": "Дообученная SpaCy", **finetuned_metrics}
])

for col in ["Precision", "Recall", "F1-score"]:
    before_after_metrics[col] = before_after_metrics[col].round(3)

before_after_metrics

,Модель,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,Базовая SpaCy,0,1,5,0.0,0.0,0.0
1,Дообученная SpaCy,3,2,2,0.6,0.6,0.6


In [169]:
from spacy import displacy

displacy.render(doc_base, style="ent", jupyter=True)

In [170]:
displacy.render(doc_finetuned, style="ent", jupyter=True)

In [171]:
df_before_after.to_csv("ner_before_after_entities.csv", index=False, encoding="utf-8-sig")
before_after_counts.to_csv("ner_before_after_counts.csv", index=False, encoding="utf-8-sig")
comparison_result.to_csv("ner_before_after_comparison.csv", index=False, encoding="utf-8-sig")
before_after_metrics.to_csv("ner_before_after_metrics.csv", index=False, encoding="utf-8-sig")

print("Результаты сравнения до и после дообучения сохранены.")

Результаты сравнения до и после дообучения сохранены.


## Пункт 9. Использование LLM для извлечения сущностей

Сравним два подхода:

| Подход      | Смысл                                                           |
| ----------- | --------------------------------------------------------------- |
| `Zero-shot` | Даём модели только инструкцию, без примеров                     |
| `Few-shot`  | Даём модели инструкцию + несколько примеров правильной разметки |


In [172]:
!pip install transformers accelerate bitsandbytes pandas -q

In [173]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

#### Zero shot

In [174]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [175]:
llm_text = """
Рисунок 1 — Методика получения МРТ изображения, набор возбуждающих импульсов.
Для уменьшения длительности исследования применяются методы параллельного сканирования.

Рисунок 2 — Пара: К-пространство и МРТ изображение.

Одним из алгоритмов параллельного сканирования является GRAPPA.
Алгоритм использует данные из автокалибровки ACS для вычисления весовых коэффициентов приемных катушек.

Современные МР-томографы используют несколько приёмных катушек с разными пространственными чувствительностями.
"""

In [176]:
zero_shot_prompt = f"""
Ты выполняешь задачу извлечения именованных сущностей из русского технического текста.

Нужно найти сущности следующих типов:
- PERSON — имена людей;
- LOCATION — географические объекты;
- ORGANIZATION — организации и учреждения;
- DATE — даты и периоды;
- WORK_OF_ART — произведения искусства;
- MISC — специальные термины, алгоритмы, технические обозначения.

Верни результат строго в виде таблицы:
Сущность | Тип | Объяснение

Текст:
{llm_text}
"""

In [177]:
zero_shot_result = llm_pipe(
    zero_shot_prompt,
    max_new_tokens=500,
    do_sample=False
)

print(zero_shot_result[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ты выполняешь задачу извлечения именованных сущностей из русского технического текста.

Нужно найти сущности следующих типов:
- PERSON — имена людей;
- LOCATION — географические объекты;
- ORGANIZATION — организации и учреждения;
- DATE — даты и периоды;
- WORK_OF_ART — произведения искусства;
- MISC — специальные термины, алгоритмы, технические обозначения.

Верни результат строго в виде таблицы:
Сущность | Тип | Объяснение

Текст:

Рисунок 1 — Методика получения МРТ изображения, набор возбуждающих импульсов.
Для уменьшения длительности исследования применяются методы параллельного сканирования.

Рисунок 2 — Пара: К-пространство и МРТ изображение.

Одним из алгоритмов параллельного сканирования является GRAPPA.
Алгоритм использует данные из автокалибровки ACS для вычисления весовых коэффициентов приемных катушек.

Современные МР-томографы используют несколько приёмных катушек с разными пространственными чувствительностями.

МР-томографы могут работать на различных частотах от 30 до 4

#### Few shot

In [178]:
few_shot_prompt = f"""
Ты выполняешь задачу извлечения именованных сущностей из русского технического текста.

Типы сущностей:
- PERSON — имена людей;
- LOCATION — географические объекты;
- ORGANIZATION — организации и учреждения;
- DATE — даты и периоды;
- WORK_OF_ART — произведения искусства;
- MISC — специальные термины, алгоритмы, технические обозначения.

Пример 1:
Текст: Алгоритм GRAPPA используется для реконструкции МРТ изображения.
Ответ:
GRAPPA | MISC | название алгоритма
МРТ изображение | MISC | технический термин

Пример 2:
Текст: Данные ACS применяются при восстановлении К-пространства.
Ответ:
ACS | MISC | техническое обозначение автокалибровочных данных
К-пространство | MISC | термин из области МРТ

Теперь выполни такую же разметку для текста ниже.

Текст:
{llm_text}

Ответ:
"""

In [179]:
few_shot_result = llm_pipe(
    few_shot_prompt,
    max_new_tokens=500,
    do_sample=False
)

print(few_shot_result[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ты выполняешь задачу извлечения именованных сущностей из русского технического текста.

Типы сущностей:
- PERSON — имена людей;
- LOCATION — географические объекты;
- ORGANIZATION — организации и учреждения;
- DATE — даты и периоды;
- WORK_OF_ART — произведения искусства;
- MISC — специальные термины, алгоритмы, технические обозначения.

Пример 1:
Текст: Алгоритм GRAPPA используется для реконструкции МРТ изображения.
Ответ:
GRAPPA | MISC | название алгоритма
МРТ изображение | MISC | технический термин

Пример 2:
Текст: Данные ACS применяются при восстановлении К-пространства.
Ответ:
ACS | MISC | техническое обозначение автокалибровочных данных
К-пространство | MISC | термин из области МРТ

Теперь выполни такую же разметку для текста ниже.

Текст:

Рисунок 1 — Методика получения МРТ изображения, набор возбуждающих импульсов.
Для уменьшения длительности исследования применяются методы параллельного сканирования.

Рисунок 2 — Пара: К-пространство и МРТ изображение.

Одним из алгоритмов п

#### Сохранение результатов

In [180]:
with open("llm_zero_shot_result.txt", "w", encoding="utf-8") as f:
    f.write(zero_shot_result[0]["generated_text"])

with open("llm_few_shot_result.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_result[0]["generated_text"])

print("Результаты LLM сохранены.")

Результаты LLM сохранены.


## Пункт 10. Итоговое сравнение подходов

Цель этапа

Нужно сравнить, какой подход оказался удобнее и практичнее:

- готовые NER-модели;
- дообучение модели;
- LLM через zero-shot и few-shot prompting.

#### Создание итоговой сравнительной таблицы

In [181]:
import pandas as pd

def safe_len(df):
    """Безопасно возвращает количество строк в DataFrame."""
    try:
        return len(df)
    except:
        return 0


def safe_unique_count(df, column):
    """Безопасно возвращает количество уникальных значений."""
    try:
        return df[column].dropna().nunique()
    except:
        return 0


def safe_entity_types(df):
    """Безопасно возвращает список типов сущностей."""
    try:
        return ", ".join(sorted(df["Тип сущности"].dropna().unique()))
    except:
        return "—"


def get_metric(metrics_df, model_name, metric_name):
    """Достаёт значение метрики из таблицы метрик."""
    try:
        value = metrics_df.loc[
            metrics_df["Модель"] == model_name,
            metric_name
        ].values[0]
        return round(float(value), 3)
    except:
        return None

In [182]:
spacy_total = safe_len(df_spacy)
stanza_total = safe_len(df_stanza)

spacy_unique = safe_unique_count(df_spacy, "Текст сущности")
stanza_unique = safe_unique_count(df_stanza, "Текст сущности")

spacy_types = safe_entity_types(df_spacy)
stanza_types = safe_entity_types(df_stanza)

print("SpaCy:")
print("Всего сущностей:", spacy_total)
print("Уникальных сущностей:", spacy_unique)
print("Типы сущностей:", spacy_types)

print("\nStanza:")
print("Всего сущностей:", stanza_total)
print("Уникальных сущностей:", stanza_unique)
print("Типы сущностей:", stanza_types)

SpaCy:
Всего сущностей: 68
Уникальных сущностей: 52
Типы сущностей: LOC, ORG, PER

Stanza:
Всего сущностей: 150
Уникальных сущностей: 127
Типы сущностей: LOC, MISC, ORG, PER


In [183]:
base_total = safe_len(df_base_entities)
finetuned_total = safe_len(df_finetuned_entities)

base_unique = safe_unique_count(df_base_entities, "Текст сущности")
finetuned_unique = safe_unique_count(df_finetuned_entities, "Текст сущности")

base_types = safe_entity_types(df_base_entities)
finetuned_types = safe_entity_types(df_finetuned_entities)

print("Базовая SpaCy:")
print("Всего сущностей:", base_total)
print("Уникальных сущностей:", base_unique)
print("Типы сущностей:", base_types)

print("\nДообученная SpaCy:")
print("Всего сущностей:", finetuned_total)
print("Уникальных сущностей:", finetuned_unique)
print("Типы сущностей:", finetuned_types)

Базовая SpaCy:
Всего сущностей: 1
Уникальных сущностей: 1
Типы сущностей: ORG

Дообученная SpaCy:
Всего сущностей: 5
Уникальных сущностей: 5
Типы сущностей: MISC


In [184]:
# Метрики для SpaCy и Stanza из этапа сравнения NER-моделей

spacy_precision = get_metric(metrics_table_report, "SpaCy", "Precision")
spacy_recall = get_metric(metrics_table_report, "SpaCy", "Recall")
spacy_f1 = get_metric(metrics_table_report, "SpaCy", "F1-score")

stanza_precision = get_metric(metrics_table_report, "Stanza", "Precision")
stanza_recall = get_metric(metrics_table_report, "Stanza", "Recall")
stanza_f1 = get_metric(metrics_table_report, "Stanza", "F1-score")

# Метрики для базовой и дообученной SpaCy

base_precision = get_metric(before_after_metrics, "Базовая SpaCy", "Precision")
base_recall = get_metric(before_after_metrics, "Базовая SpaCy", "Recall")
base_f1 = get_metric(before_after_metrics, "Базовая SpaCy", "F1-score")

finetuned_precision = get_metric(before_after_metrics, "Дообученная SpaCy", "Precision")
finetuned_recall = get_metric(before_after_metrics, "Дообученная SpaCy", "Recall")
finetuned_f1 = get_metric(before_after_metrics, "Дообученная SpaCy", "F1-score")

In [185]:
def get_llm_output(result):
    try:
        return result[0]["generated_text"]
    except Exception:
        return ""

zero_shot_text = get_llm_output(zero_shot_result) if "zero_shot_result" in globals() else ""
few_shot_text = get_llm_output(few_shot_result) if "few_shot_result" in globals() else ""

print("Zero-shot output:")
print(zero_shot_text[-1500:])

print("\nFew-shot output:")
print(few_shot_text[-1500:])

Zero-shot output:
тирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под различные условия использования.

GRAPPA может быть адаптирована под раз

In [186]:
import re
import pandas as pd

allowed_labels = {
    "PERSON", "PER",
    "LOCATION", "LOC",
    "ORGANIZATION", "ORG",
    "DATE",
    "WORK_OF_ART",
    "MISC"
}

label_mapping = {
    "PERSON": "PER",
    "PER": "PER",
    "LOCATION": "LOC",
    "LOC": "LOC",
    "ORGANIZATION": "ORG",
    "ORG": "ORG",
    "DATE": "DATE",
    "WORK_OF_ART": "WORK_OF_ART",
    "MISC": "MISC"
}

def parse_llm_entities(output_text, approach_name):
    rows = []

    for line in output_text.splitlines():
        line = line.strip()

        if "|" not in line:
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) < 2:
            continue

        entity = parts[0]
        label = parts[1].upper()

        if entity.lower() in ["сущность", "entity", "ответ", "текст"]:
            continue

        if label not in allowed_labels:
            continue

        rows.append({
            "Подход": approach_name,
            "Текст сущности": entity,
            "Тип сущности": label_mapping.get(label, label),
            "Объяснение": parts[2] if len(parts) > 2 else ""
        })

    return pd.DataFrame(rows)

df_zero_shot_llm = parse_llm_entities(zero_shot_text, "LLM zero-shot")
df_few_shot_llm = parse_llm_entities(few_shot_text, "LLM few-shot")

llm_results_table = pd.concat(
    [df_zero_shot_llm, df_few_shot_llm],
    ignore_index=True
)

llm_results_table

,Подход,Текст сущности,Тип сущности,Объяснение
0,LLM few-shot,GRAPPA,MISC,название алгоритма
1,LLM few-shot,МРТ изображение,MISC,технический термин
2,LLM few-shot,ACS,MISC,техническое обозначение автокалибровочных данных
3,LLM few-shot,К-пространство,MISC,термин из области МРТ
4,LLM few-shot,Рисунок 1,MISC,техническая диаграмма
5,LLM few-shot,МРТ изображение,MISC,технический термин
6,LLM few-shot,К-пространство,MISC,термин из области МРТ
7,LLM few-shot,PARA,MISC,пара (математическая концепция)
8,LLM few-shot,GRAPPA,ORG,алгоритм
9,LLM few-shot,ACS,MISC,техническое обозначение автокалибровочных данных


In [187]:
print("df_zero_shot_llm columns:", df_zero_shot_llm.columns.tolist())
print("df_few_shot_llm columns:", df_few_shot_llm.columns.tolist())

display(df_zero_shot_llm.head())
display(df_few_shot_llm.head())

df_zero_shot_llm columns: []
df_few_shot_llm columns: ['Подход', 'Текст сущности', 'Тип сущности', 'Объяснение']


""


,Подход,Текст сущности,Тип сущности,Объяснение
0,LLM few-shot,GRAPPA,MISC,название алгоритма
1,LLM few-shot,МРТ изображение,MISC,технический термин
2,LLM few-shot,ACS,MISC,техническое обозначение автокалибровочных данных
3,LLM few-shot,К-пространство,MISC,термин из области МРТ
4,LLM few-shot,Рисунок 1,MISC,техническая диаграмма


In [188]:
def prepare_entities_for_metrics_auto(df):
    prepared = df.copy()

    # Если таблица пустая и без колонок — создаём нужные колонки
    if "Текст сущности" not in prepared.columns:
        prepared["Текст сущности"] = ""

    if "Тип сущности" not in prepared.columns:
        prepared["Тип сущности"] = ""

    prepared["Текст сущности"] = (
        prepared["Текст сущности"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    prepared["Тип сущности"] = (
        prepared["Тип сущности"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    prepared["Тип сущности"] = prepared["Тип сущности"].map(
        lambda x: label_mapping.get(x, x)
    )

    prepared = prepared[
        (prepared["Текст сущности"] != "") &
        (prepared["Тип сущности"] != "")
    ]

    return prepared[["Текст сущности", "Тип сущности"]].drop_duplicates()

In [189]:
def calculate_entity_metrics_auto(gold_df, pred_df):
    gold_set = set(zip(gold_df["Текст сущности"], gold_df["Тип сущности"]))
    pred_set = set(zip(pred_df["Текст сущности"], pred_df["Тип сущности"]))

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "True Positive": tp,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1-score": round(f1, 3)
    }


# Берём эталонную разметку автоматически из уже существующих переменных
if "df_gold_test" in globals():
    gold_for_llm = df_gold_test
elif "df_gold" in globals():
    gold_for_llm = df_gold
else:
    gold_for_llm = pd.DataFrame(columns=["Текст сущности", "Тип сущности"])

gold_for_llm_prepared = prepare_entities_for_metrics_auto(gold_for_llm)

zero_llm_prepared = prepare_entities_for_metrics_auto(df_zero_shot_llm)
few_llm_prepared = prepare_entities_for_metrics_auto(df_few_shot_llm)

zero_shot_metrics = calculate_entity_metrics_auto(
    gold_for_llm_prepared,
    zero_llm_prepared
)

few_shot_metrics = calculate_entity_metrics_auto(
    gold_for_llm_prepared,
    few_llm_prepared
)

llm_metrics_table = pd.DataFrame([
    {"Подход": "LLM zero-shot", **zero_shot_metrics},
    {"Подход": "LLM few-shot", **few_shot_metrics}
])

llm_metrics_table

,Подход,True Positive,False Positive,False Negative,Precision,Recall,F1-score
0,LLM zero-shot,0,0,5,0.000,0.0,0.000
1,LLM few-shot,1,11,4,0.083,0.2,0.118


In [190]:
def calculate_llm_accuracy(df, approach_name):
    try:
        subset = df[df["Подход"] == approach_name]

        if len(subset) == 0:
            return None

        correct = len(subset[subset["Оценка"].str.lower() == "верно"])
        total = len(subset)

        return round(correct / total, 3)
    except:
        return None


def calculate_llm_total(df, approach_name):
    try:
        return len(df[df["Подход"] == approach_name])
    except:
        return 0


zero_shot_total = calculate_llm_total(llm_results_table, "Zero-shot")
few_shot_total = calculate_llm_total(llm_results_table, "Few-shot")

zero_shot_accuracy = calculate_llm_accuracy(llm_results_table, "Zero-shot")
few_shot_accuracy = calculate_llm_accuracy(llm_results_table, "Few-shot")

print("Zero-shot сущностей:", zero_shot_total)
print("Zero-shot accuracy:", zero_shot_accuracy)

print("Few-shot сущностей:", few_shot_total)
print("Few-shot accuracy:", few_shot_accuracy)

Zero-shot сущностей: 0
Zero-shot accuracy: None
Few-shot сущностей: 0
Few-shot accuracy: None


In [191]:
final_auto_comparison = pd.DataFrame([
    {
        "Подход": "SpaCy baseline",
        "Всего найдено сущностей": spacy_total,
        "Уникальных сущностей": spacy_unique,
        "Типы сущностей": spacy_types,
        "Precision": spacy_precision,
        "Recall": spacy_recall,
        "F1-score": spacy_f1,
        "Комментарий": "Готовая модель SpaCy без дополнительного обучения"
    },
    {
        "Подход": "Stanza baseline",
        "Всего найдено сущностей": stanza_total,
        "Уникальных сущностей": stanza_unique,
        "Типы сущностей": stanza_types,
        "Precision": stanza_precision,
        "Recall": stanza_recall,
        "F1-score": stanza_f1,
        "Комментарий": "Готовая модель Stanza без дополнительного обучения"
    },
    {
        "Подход": "Базовая SpaCy на тестовом фрагменте",
        "Всего найдено сущностей": base_total,
        "Уникальных сущностей": base_unique,
        "Типы сущностей": base_types,
        "Precision": base_precision,
        "Recall": base_recall,
        "F1-score": base_f1,
        "Комментарий": "Исходная SpaCy до fine-tuning"
    },
    {
        "Подход": "Дообученная SpaCy",
        "Всего найдено сущностей": finetuned_total,
        "Уникальных сущностей": finetuned_unique,
        "Типы сущностей": finetuned_types,
        "Precision": finetuned_precision,
        "Recall": finetuned_recall,
        "F1-score": finetuned_f1,
        "Комментарий": "SpaCy после дообучения на технических терминах"
    },
    {
        "Подход": "LLM zero-shot",
        "Всего найдено сущностей": zero_shot_total,
        "Уникальных сущностей": zero_shot_total,
        "Типы сущностей": "по ответу LLM",
        "Precision": zero_shot_accuracy,
        "Recall": None,
        "F1-score": None,
        "Комментарий": "Оценка рассчитана по вручную отмеченным верным ответам"
    },
    {
        "Подход": "LLM few-shot",
        "Всего найдено сущностей": few_shot_total,
        "Уникальных сущностей": few_shot_total,
        "Типы сущностей": "по ответу LLM",
        "Precision": few_shot_accuracy,
        "Recall": None,
        "F1-score": None,
        "Комментарий": "Оценка рассчитана по вручную отмеченным верным ответам"
    }
])

final_auto_comparison

,Подход,Всего найдено сущностей,Уникальных сущностей,Типы сущностей,Precision,Recall,F1-score,Комментарий
0,SpaCy baseline,68,52,"LOC, ORG, PER",0.000,0.000,0.000,Готовая модель SpaCy без дополнительного обучения
1,Stanza baseline,150,127,"LOC, MISC, ORG, PER",0.023,0.375,0.044,Готовая модель Stanza без дополнительного обуч...
2,Базовая SpaCy на тестовом фрагменте,1,1,ORG,0.000,0.000,0.000,Исходная SpaCy до fine-tuning
3,Дообученная SpaCy,5,5,MISC,0.600,0.600,0.600,SpaCy после дообучения на технических терминах
4,LLM zero-shot,0,0,по ответу LLM,NaN,NaN,NaN,Оценка рассчитана по вручную отмеченным верным...
5,LLM few-shot,0,0,по ответу LLM,NaN,NaN,NaN,Оценка рассчитана по вручную отмеченным верным...


In [192]:
models_with_f1 = final_auto_comparison.dropna(subset=["F1-score"])

if len(models_with_f1) > 0:
    best_model = models_with_f1.loc[models_with_f1["F1-score"].idxmax()]

    print("Лучшая модель по F1-score:")
    print(best_model["Подход"])
    print("F1-score:", best_model["F1-score"])
else:
    print("Нет данных F1-score для сравнения.")

Лучшая модель по F1-score:
Дообученная SpaCy
F1-score: 0.6
